In [ ]:
import pandas as pd
import numpy as np


In [ ]:
df = pd.read_csv('data/fraud_oracle.csv')
print(df.columns)
print(df.dtypes)

missing_values = df.isnull().sum()
missing_summary = missing_values[missing_values > 0]
if missing_summary.empty:
    print("No missing values found in the dataset.")
else:
    print("Missing values detected:")
    print(missing_summary)

In [ ]:
df

In [ ]:
'''
Yes-No columns change to 1-0 
change ordered columns to numerical values
One hot encoding for columns with multiple values but not ordered
'''

yes_no_columns = ['PoliceReportFiled', 'WitnessPresent']
df[yes_no_columns] = df[yes_no_columns].apply(lambda x: x.map({'Yes': 1, 'No': 0}))


ordinal_mappings = {
    'AgeOfVehicle': {'less than 1': 0, '1 year': 1, '2 years': 2, '3 years': 3, '4 years': 4, 
                     '5 years': 5, '6 years': 6, '7 years': 7, 'more than 7': 8},

    'AgeOfPolicyHolder': {'16 to 17': 16, '18 to 20': 19, '21 to 25': 23, '26 to 30': 28, 
                          '31 to 35': 33, '36 to 40': 38, '41 to 50': 45, '51 to 65': 58, 
                          'over 65': 70},

    'NumberOfCars': {'1 vehicle': 1, '2 vehicles': 2, '3 to 4': 3, '5 to 8': 5, 'more than 8': 9},

    'NumberOfSuppliments': {'none': 0, '1 to 2': 1, '3 to 5': 3, 'more than 5': 5},

    'AddressChange_Claim': {'no change': 0, 'under 6 months': 1, '1 year': 2, 
                             '2 to 3 years': 3, '4 to 8 years': 4, 'more than 8 years': 5},

    'Days_Policy_Accident': {'none': 0, '1 to 7': 1, '8 to 15': 2, '15 to 30': 3, "more than 30": 4},

    'Days_Policy_Claim': {'none': 0, '8 to 15': 1, '15 to 30': 2, 'more than 30': 3},
    
    'PastNumberOfClaims': {'none': 0, '1': 1, '2 to 4': 2, 'more than 4': 3},

    'AgeOfVehicle': { 'new': 0, '2 years': 1, '3 years': 2, '4 years': 3, '5 years': 4, '6 years': 5, '7 years': 6, 'more than 7': 7},
                     
    'VehiclePrice': {'less than 20000': 0, '20000 to 29000': 1, '30000 to 39000': 2, '40000 to 59000': 3, '60000 to 69000': 4, 'more than 69000': 5}
}


# test = df["MonthClaimed"].unique()
# array(['Jan', 'Nov', 'Jul', 'Feb', 'Mar', 'Dec', 'Apr', 'Aug', 'May',
#        'Jun', 'Sep', 'Oct', '0'], dtype=object)|

for col, mapping in ordinal_mappings.items():
    df[col] = df[col].map(mapping)


binary_mappings = {
    'Sex': {'Male': 1, 'Female': 0},
    'AgentType': {'Internal': 1, 'External': 0}
}

for col, mapping in binary_mappings.items():
    df[col] = df[col].map(mapping)

df = pd.get_dummies(df, columns=['MaritalStatus'], drop_first=True)

# df.drop(columns=['PolicyNumber', 'RepNumber'], inplace=True)


print(df)

In [ ]:
'''
feature engineering: 
1. ClaimsPerPolicy - Counts the number of claims made by each policyholder.
2. DaysSinceLastClaim - Calculates the time gap between consecutive claims for the same policyholder.
3. UnusualDeductible - Flags claims with deductible amounts above the median.
4. AccidentAreaChange - Identifies policyholders with claims in different accident areas.
5. ClaimsPerAgent - Counts the number of claims handled by each agent type.
'''

df['ClaimsPerPolicy'] = df.groupby('PolicyNumber')['PolicyNumber'].transform('count')

df = df.sort_values(by=['PolicyNumber', 'Year', 'Month', 'WeekOfMonthClaimed'])

# there is 1 entry where month claimed is 0 and not any month, so i will replace it with 0 to show that maybe it has not been claimed?
month_mapping = {
    'Jan': 1, 'Feb': 2, 'Mar': 3, 'Apr': 4, 'May': 5, 'Jun': 6,
    'Jul': 7, 'Aug': 8, 'Sep': 9, 'Oct': 10, 'Nov': 11, 'Dec': 12,
    '0': 0
}
df['MonthNumeric'] = df['MonthClaimed'].map(month_mapping)

df['DaysSinceLastClaim'] = df.groupby('PolicyNumber')['WeekOfMonthClaimed'].diff().fillna(0)

df['UnusualDeductible'] = (df['Deductible'] > df['Deductible'].median()).astype(int)

df['AccidentAreaChange'] = df.groupby('PolicyNumber')['AccidentArea'].nunique() > 1
df['AccidentAreaChange'] = df['AccidentAreaChange'].fillna(False).astype(int)

df['ClaimsPerAgent'] = df.groupby('AgentType')['AgentType'].transform('count')

df


In [ ]:
missing_values = df.isnull().sum()
missing_summary = missing_values[missing_values > 0]
if missing_summary.empty:
    print("No missing values found in the dataset.")
else:
    print("Missing values detected:")
    print(missing_summary)

In [ ]:
feature_columns = [
    'ClaimsPerPolicy', 'DaysSinceLastClaim', 'UnusualDeductible', 
    'AccidentAreaChange', 'ClaimsPerAgent'
]
feature_stats = df[feature_columns].describe().T
print(feature_stats)

In [ ]:
# since sd of ClaimsPerPolicy, DaysSinceLastClaim, AccidentAreaChange is 0, maybe we can conclude that policy number is unique?
# Month claimed is dropped as it has been converted to monthnumeric, rename back to monthclaimed
df.drop(columns=['AccidentAreaChange', 'DaysSinceLastClaim', 'ClaimsPerPolicy', "MonthClaimed"], inplace=True)
df.rename(columns={"MonthNumeric": "MonthClaimed"}, inplace=True)
# df.to_csv('data/fraud_oracle_processed.csv')

# Forward and backwards selection

In [ ]:
!pip install scikit-learn

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.compose import ColumnTransformer

df = pd.read_csv("data/fraud_oracle_processed.csv")
df = df.drop(columns=["Unnamed: 0", "PolicyNumber", "RepNumber"])
X = df.drop(columns=["FraudFound_P"])
y = df["FraudFound_P"]

categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
numeric_cols = X.select_dtypes(include=["int64", "bool"]).columns.tolist()

preprocessor = ColumnTransformer(transformers=[
    ("num", StandardScaler(), numeric_cols),
    ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_cols)
])

X_processed = preprocessor.fit_transform(X)
feature_names = preprocessor.get_feature_names_out()

X_train, X_test, y_train, y_test = train_test_split(X_processed, y, test_size=0.2, random_state=42)

# Logistic
log_reg = LogisticRegression(max_iter=1000)
forward_log = SequentialFeatureSelector(log_reg,
                                        n_features_to_select=10,
                                        direction="forward",
                                        scoring="accuracy",
                                        cv=5,
                                        n_jobs=-1)
forward_log.fit(X_train, y_train)
forward_log_features = feature_names[forward_log.get_support()]


print("\nTop 10 Features (Forward Selection - Logistic Regression):")
for i, feature in enumerate(forward_log_features, 1):
    print(f"{i}. {feature}")

In [ ]:
backward_log = SequentialFeatureSelector(log_reg,
                                         n_features_to_select=10,
                                         direction="backward",
                                         scoring="accuracy",
                                         cv=5,
                                         n_jobs=-1)
backward_log.fit(X_train, y_train)
backward_log_features = feature_names[backward_log.get_support()]


print("\nTop 10 Features (Backward Selection - Logistic Regression):")
for i, feature in enumerate(backward_log_features, 1):
    print(f"{i}. {feature}")

In [ ]:
# random forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
forward_rf = SequentialFeatureSelector(rf,
                                       n_features_to_select=10,
                                       direction="forward",
                                       scoring="accuracy",
                                       cv=5,
                                       n_jobs=-1)
forward_rf.fit(X_train, y_train)
forward_rf_features = feature_names[forward_rf.get_support()]



print("\nTop 10 Features (Forward Selection - Random Forest):")
for i, feature in enumerate(forward_rf_features, 1):
    print(f"{i}. {feature}")

In [ ]:
# Backward
backward_rf = SequentialFeatureSelector(rf,
                                        n_features_to_select=10,
                                        direction="backward",
                                        scoring="accuracy",
                                        cv=5,
                                        n_jobs=-1)
backward_rf.fit(X_train, y_train)
backward_rf_features = feature_names[backward_rf.get_support()]

print("\nTop 10 Features (Backward Selection - Random Forest):")
for i, feature in enumerate(backward_rf_features, 1):
    print(f"{i}. {feature}")